#(TSLA) — Comprehensive Financial Analysis

> **Author:** Financial Analysis Project  
> **Data Source:** Yahoo Finance (via `yfinance`) | SEC Filings  
> **Period:** 2019 – 2024

---

## 📋 Table of Contents
1. [Setup & Imports](#1-setup)
2. [Data Loading](#2-data)
3. [Technical Indicators](#3-indicators)
4. [Price & Volume Analysis](#4-price)
5. [Returns & Risk Analysis](#5-returns)
6. [Annual Financial Highlights](#6-financials)
7. [Monte Carlo Simulation](#7-monte-carlo)
8. [Summary & Key Metrics](#8-summary)

---
## 1. Setup & Imports <a id='1-setup'></a>

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install yfinance pandas numpy matplotlib seaborn scipy

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

try:
    import yfinance as yf
    YFINANCE_AVAILABLE = True
    print('✅ yfinance available — will use live data')
except ImportError:
    YFINANCE_AVAILABLE = False
    print('⚠️  yfinance not found — using synthetic data')

# ── Style ──────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#F8F9FA',
    'axes.facecolor':   '#FFFFFF',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'font.family':      'DejaVu Sans',
})

COLORS = {
    'primary':   '#E31937',   # Tesla red
    'secondary': '#1A1A1A',
    'accent':    '#5C6BC0',
    'positive':  '#26A69A',
    'negative':  '#EF5350',
    'neutral':   '#78909C',
}

# ── Config ─────────────────────────────────────────────────────
TICKER         = 'TSLA'
START_DATE     = '2019-01-01'
END_DATE       = '2024-12-31'
MC_SIMULATIONS = 1000
MC_DAYS        = 252   # 1 trading year

print(f'\nConfiguration:')
print(f'  Ticker : {TICKER}')
print(f'  Period : {START_DATE} → {END_DATE}')
print(f'  MC Runs: {MC_SIMULATIONS:,} × {MC_DAYS} days')

---
## 2. Data Loading <a id='2-data'></a>

In [ ]:
def load_stock_data(ticker, start, end):
    """Download OHLCV data from Yahoo Finance, or generate synthetic data."""
    if YFINANCE_AVAILABLE:
        print(f'Downloading {ticker} from Yahoo Finance...')
        df = yf.download(ticker, start=start, end=end, progress=False)
        df.columns = [c[0] if isinstance(c, tuple) else c for c in df.columns]
    else:
        print('Generating synthetic TSLA-like price data...')
        dates = pd.date_range(start=start, end=end, freq='B')
        np.random.seed(42)
        n = len(dates)
        returns = np.random.normal(0.001, 0.035, n)
        price = 60.0 * np.exp(np.cumsum(returns))
        df = pd.DataFrame({
            'Open':   price * np.random.uniform(0.98, 1.00, n),
            'High':   price * np.random.uniform(1.00, 1.04, n),
            'Low':    price * np.random.uniform(0.96, 1.00, n),
            'Close':  price,
            'Volume': np.random.randint(50_000_000, 200_000_000, n).astype(float),
        }, index=dates)
    df = df.dropna()
    print(f'  ✅ Loaded {len(df):,} trading days ({df.index[0].date()} → {df.index[-1].date()})')
    return df

df = load_stock_data(TICKER, START_DATE, END_DATE)
df.tail(5)

---
## 3. Technical Indicators <a id='3-indicators'></a>

We compute the following indicators:
- **SMA** 20 / 50 / 200-day Simple Moving Averages
- **EMA** 12 / 26-day Exponential MAs → **MACD**
- **Bollinger Bands** (20-day, ±2σ)
- **RSI** (14-day Relative Strength Index)
- **Daily / Log Returns**, Cumulative Return, Rolling Volatility

In [ ]:
def add_indicators(df):
    close = df['Close']

    # Moving averages
    df['SMA_20']  = close.rolling(20).mean()
    df['SMA_50']  = close.rolling(50).mean()
    df['SMA_200'] = close.rolling(200).mean()
    df['EMA_12']  = close.ewm(span=12, adjust=False).mean()
    df['EMA_26']  = close.ewm(span=26, adjust=False).mean()
    df['MACD']    = df['EMA_12'] - df['EMA_26']
    df['Signal']  = df['MACD'].ewm(span=9, adjust=False).mean()

    # Bollinger Bands
    df['BB_Mid']   = close.rolling(20).mean()
    df['BB_Std']   = close.rolling(20).std()
    df['BB_Upper'] = df['BB_Mid'] + 2 * df['BB_Std']
    df['BB_Lower'] = df['BB_Mid'] - 2 * df['BB_Std']

    # RSI (14-day)
    delta = close.diff()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))

    # Returns
    df['Daily_Return']  = close.pct_change()
    df['Cum_Return']    = (1 + df['Daily_Return']).cumprod() - 1
    df['Log_Return']    = np.log(close / close.shift(1))
    df['Volatility_30'] = df['Daily_Return'].rolling(30).std() * np.sqrt(252)

    return df

df = add_indicators(df)
print('Indicator columns added:')
indicator_cols = ['SMA_20','SMA_50','SMA_200','MACD','Signal',
                  'BB_Upper','BB_Lower','RSI','Daily_Return','Volatility_30']
print(df[indicator_cols].tail(3).round(2).to_string())

---
## 4. Price & Volume Technical Dashboard <a id='4-price'></a>

In [ ]:
fig = plt.figure(figsize=(16, 14))
fig.suptitle('Tesla (TSLA) — Technical Analysis Dashboard',
             fontsize=18, fontweight='bold', color=COLORS['secondary'], y=0.98)
gs = gridspec.GridSpec(4, 1, figure=fig, hspace=0.05,
                       height_ratios=[3, 1, 1, 1])

# ── Price + Bollinger + SMAs ──────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
ax1.plot(df.index, df['Close'],   color=COLORS['primary'],  lw=1.5, label='Close')
ax1.plot(df.index, df['SMA_20'],  color=COLORS['accent'],   lw=1.0, ls='--', label='SMA 20')
ax1.plot(df.index, df['SMA_50'],  color='orange',            lw=1.0, ls='--', label='SMA 50')
ax1.plot(df.index, df['SMA_200'], color='purple',            lw=1.0, ls='--', label='SMA 200')
ax1.fill_between(df.index, df['BB_Upper'], df['BB_Lower'],
                 alpha=0.08, color=COLORS['accent'], label='Bollinger Bands')
ax1.plot(df.index, df['BB_Upper'], color=COLORS['accent'], lw=0.6, ls=':')
ax1.plot(df.index, df['BB_Lower'], color=COLORS['accent'], lw=0.6, ls=':')
ax1.set_ylabel('Price (USD)', fontweight='bold')
ax1.legend(loc='upper left', fontsize=8, ncol=3)
ax1.set_xticklabels([])

# ── Volume ────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[1], sharex=ax1)
vol_colors = [COLORS['positive'] if r >= 0 else COLORS['negative']
              for r in df['Daily_Return'].fillna(0)]
ax2.bar(df.index, df['Volume'] / 1e6, color=vol_colors, alpha=0.7, width=1)
ax2.set_ylabel('Volume (M)', fontweight='bold')
ax2.set_xticklabels([])

# ── RSI ───────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[2], sharex=ax1)
ax3.plot(df.index, df['RSI'], color=COLORS['accent'], lw=1.2)
ax3.axhline(70, color=COLORS['negative'], ls='--', lw=0.8, alpha=0.8, label='Overbought (70)')
ax3.axhline(30, color=COLORS['positive'], ls='--', lw=0.8, alpha=0.8, label='Oversold (30)')
ax3.fill_between(df.index, df['RSI'], 70, where=(df['RSI'] >= 70), alpha=0.2, color=COLORS['negative'])
ax3.fill_between(df.index, df['RSI'], 30, where=(df['RSI'] <= 30), alpha=0.2, color=COLORS['positive'])
ax3.set_ylim(0, 100)
ax3.set_ylabel('RSI (14)', fontweight='bold')
ax3.legend(fontsize=7, loc='upper left')
ax3.set_xticklabels([])

# ── MACD ──────────────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[3], sharex=ax1)
macd_hist = df['MACD'] - df['Signal']
ax4.plot(df.index, df['MACD'],   color=COLORS['primary'], lw=1.2, label='MACD')
ax4.plot(df.index, df['Signal'], color='orange',           lw=1.0, ls='--', label='Signal')
ax4.bar(df.index, macd_hist,
        color=[COLORS['positive'] if v >= 0 else COLORS['negative'] for v in macd_hist.fillna(0)],
        alpha=0.5, width=1)
ax4.axhline(0, color='black', lw=0.5)
ax4.set_ylabel('MACD', fontweight='bold')
ax4.legend(fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

---
## 5. Returns & Risk Analysis <a id='5-returns'></a>

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Tesla — Returns & Risk Analysis', fontsize=15,
             fontweight='bold', color=COLORS['secondary'])

# Daily return histogram
ax = axes[0]
returns = df['Daily_Return'].dropna()
ax.hist(returns, bins=80, color=COLORS['primary'], alpha=0.7, edgecolor='white', lw=0.3)
mu, sigma = returns.mean(), returns.std()
x = np.linspace(returns.min(), returns.max(), 200)
ax.plot(x, stats.norm.pdf(x, mu, sigma) * len(returns) * (returns.max()-returns.min()) / 80,
        color=COLORS['secondary'], lw=2, label=f'Normal fit\nμ={mu:.4f}  σ={sigma:.4f}')
ax.set_title('Daily Return Distribution', fontweight='bold')
ax.set_xlabel('Daily Return')
ax.set_ylabel('Frequency')
ax.legend(fontsize=8)

# Cumulative return
ax = axes[1]
ax.plot(df.index, df['Cum_Return'] * 100, color=COLORS['primary'], lw=1.5)
ax.fill_between(df.index, df['Cum_Return'] * 100, alpha=0.15, color=COLORS['primary'])
ax.axhline(0, color='black', lw=0.5)
ax.set_title('Cumulative Return (%)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Return (%)')

# Rolling volatility
ax = axes[2]
ax.plot(df.index, df['Volatility_30'] * 100, color=COLORS['accent'], lw=1.2)
ax.fill_between(df.index, df['Volatility_30'] * 100, alpha=0.15, color=COLORS['accent'])
ax.set_title('30-Day Rolling Volatility (Annualised)', fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Volatility (%)')

plt.tight_layout()
plt.show()

In [ ]:
# ── Descriptive Risk Statistics ────────────────────────────────────────────────
annual_return = (1 + returns.mean()) ** 252 - 1
annual_vol    = returns.std() * np.sqrt(252)
sharpe        = annual_return / annual_vol
skewness      = float(stats.skew(returns))
kurt          = float(stats.kurtosis(returns))
var_95        = float(np.percentile(returns, 5))
cvar_95       = float(returns[returns <= var_95].mean())
max_dd        = float(((df['Close'] / df['Close'].cummax()) - 1).min())

risk_summary = pd.DataFrame({
    'Metric': [
        'Annual Return (CAGR)', 'Annual Volatility', 'Sharpe Ratio',
        'Skewness', 'Excess Kurtosis', 'VaR 95% (daily)',
        'CVaR 95% (daily)', 'Maximum Drawdown'
    ],
    'Value': [
        f'{annual_return*100:.2f}%', f'{annual_vol*100:.2f}%', f'{sharpe:.2f}',
        f'{skewness:.3f}', f'{kurt:.3f}', f'{var_95*100:.2f}%',
        f'{cvar_95*100:.2f}%', f'{max_dd*100:.2f}%'
    ]
})
risk_summary

---
## 6. Annual Financial Highlights <a id='6-financials'></a>

Data sourced from Tesla SEC filings and quarterly earnings reports (2018–2023).

In [ ]:
fin = pd.DataFrame({
    'Year':            [2018,  2019,  2020,  2021,  2022,   2023],
    'Revenue_B':       [21.46, 24.58, 31.54, 53.82, 81.46,  96.77],
    'GrossProfit_B':   [4.46,  4.07,  6.63,  13.66, 20.85,  17.66],
    'NetIncome_B':     [-0.98, -0.86, 0.72,  5.52,  12.56,  14.99],
    'FreeCashFlow_B':  [-1.00, -1.00, 2.79,  4.98,  7.57,   4.36],
    'GrossMargin_Pct': [20.8,  16.6,  21.0,  25.4,  25.6,   18.2],
    'Deliveries_K':    [245,   367,   500,   936,   1314,   1809],
})

print('Tesla Annual Financial Data:')
fin.set_index('Year')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Tesla — Annual Financial Highlights (2018–2023)',
             fontsize=15, fontweight='bold', color=COLORS['secondary'])

metrics = [
    ('Revenue_B',       'Revenue ($ Billion)',        COLORS['primary']),
    ('GrossProfit_B',   'Gross Profit ($ Billion)',   COLORS['accent']),
    ('NetIncome_B',     'Net Income ($ Billion)',      None),
    ('FreeCashFlow_B',  'Free Cash Flow ($ Billion)',  None),
    ('GrossMargin_Pct', 'Gross Margin (%)',             'purple'),
    ('Deliveries_K',    'Vehicle Deliveries (000s)',   COLORS['positive']),
]

for ax, (col, title, color) in zip(axes.flat, metrics):
    values = fin[col]
    bar_colors = ([COLORS['positive'] if v >= 0 else COLORS['negative'] for v in values]
                  if color is None else color)
    ax.bar(fin['Year'], values, color=bar_colors, alpha=0.85, edgecolor='white', lw=0.5)
    for year, val in zip(fin['Year'], values):
        ax.text(year, val + abs(val)*0.03, f'{val:.1f}',
                ha='center', va='bottom' if val >= 0 else 'top',
                fontsize=8, fontweight='bold')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Year')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xticks(fin['Year'])

plt.tight_layout()
plt.show()

In [ ]:
# Year-over-Year Growth Rates
growth = fin.set_index('Year')[['Revenue_B', 'GrossProfit_B', 'Deliveries_K']].pct_change() * 100
growth.columns = ['Revenue Growth %', 'Gross Profit Growth %', 'Delivery Growth %']
print('Year-over-Year Growth Rates:')
growth.round(1)

---
## 7. Monte Carlo Price Simulation <a id='7-monte-carlo'></a>

We use **Geometric Brownian Motion (GBM)** to simulate 1,000 possible price paths over the next 252 trading days (~1 year).

$$S_{t+1} = S_t \cdot e^{(\mu - \frac{\sigma^2}{2})\Delta t + \sigma \sqrt{\Delta t}\, Z}$$

where $Z \sim \mathcal{N}(0,1)$, $\mu$ = mean log return, $\sigma$ = std of log returns.

In [ ]:
def monte_carlo(df, simulations=1000, days=252):
    """GBM-based Monte Carlo price simulation."""
    log_returns = df['Log_Return'].dropna()
    mu    = log_returns.mean()
    sigma = log_returns.std()
    last  = float(df['Close'].iloc[-1])

    print(f'  μ (daily log return) = {mu:.5f}')
    print(f'  σ (daily log vol)    = {sigma:.5f}')
    print(f'  Starting price       = ${last:,.2f}')
    print(f'  Running {simulations:,} simulations × {days} days...')

    results = np.zeros((days, simulations))
    for s in range(simulations):
        prices = [last]
        for _ in range(days - 1):
            shock = np.random.normal(mu, sigma)
            prices.append(prices[-1] * np.exp(shock))
        results[:, s] = prices
    return results

np.random.seed(42)
mc_results = monte_carlo(df, MC_SIMULATIONS, MC_DAYS)
print('  ✅ Simulation complete')

In [ ]:
last_price   = float(df['Close'].iloc[-1])
final_prices = mc_results[-1, :]
p5, p25, p50, p75, p95 = np.percentile(final_prices, [5, 25, 50, 75, 95])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'Tesla — Monte Carlo Simulation ({MC_SIMULATIONS:,} paths, {MC_DAYS}-day horizon)',
             fontsize=15, fontweight='bold', color=COLORS['secondary'])

# Simulation paths
ax = axes[0]
days_x = np.arange(MC_DAYS)
for i in range(min(200, MC_SIMULATIONS)):
    ax.plot(days_x, mc_results[:, i], alpha=0.04, color=COLORS['primary'], lw=0.5)
ax.plot(days_x, np.percentile(mc_results, 95, axis=1),
        color=COLORS['positive'], lw=2, ls='--', label='95th percentile')
ax.plot(days_x, np.percentile(mc_results, 50, axis=1),
        color=COLORS['secondary'], lw=2, label='Median (50th)')
ax.plot(days_x, np.percentile(mc_results,  5, axis=1),
        color=COLORS['negative'], lw=2, ls='--', label='5th percentile')
ax.axhline(last_price, color='black', lw=1.2, ls=':', label=f'Current ${last_price:,.2f}')
ax.set_xlabel('Trading Days')
ax.set_ylabel('Simulated Price (USD)')
ax.set_title('Simulated Price Paths', fontweight='bold')
ax.legend(fontsize=9)

# Final price distribution
ax = axes[1]
ax.hist(final_prices, bins=60, color=COLORS['primary'], alpha=0.75, edgecolor='white', lw=0.3)
for pct, label, color in [
    (p5,  f'Bear  5th  ${p5:,.0f}',  COLORS['negative']),
    (p50, f'Base  50th ${p50:,.0f}', COLORS['secondary']),
    (p95, f'Bull  95th ${p95:,.0f}', COLORS['positive']),
]:
    ax.axvline(pct, color=color, lw=2, ls='--', label=label)
ax.axvline(last_price, color='black', lw=1.5, ls=':', label=f'Current ${last_price:,.0f}')
ax.set_xlabel('Price (USD)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Simulated Final Prices', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print(f'\n📊 Monte Carlo Forecast Summary (1-year horizon):')
print(f'  Bear Case  (5th pct) : ${p5:>10,.2f}')
print(f'  Low  Case (25th pct) : ${p25:>10,.2f}')
print(f'  Base Case (50th pct) : ${p50:>10,.2f}')
print(f'  High Case (75th pct) : ${p75:>10,.2f}')
print(f'  Bull Case (95th pct) : ${p95:>10,.2f}')

---
## 8. Summary & Key Metrics <a id='8-summary'></a>

In [ ]:
print('═'*50)
print('  TESLA (TSLA) — COMPLETE ANALYSIS SUMMARY')
print('═'*50)

summary = {
    '── PRICE ──────────────────────────────': '',
    'Current Price':          f'${float(df["Close"].iloc[-1]):,.2f}',
    'Period High':            f'${df["High"].max():,.2f}',
    'Period Low':             f'${df["Low"].min():,.2f}',
    '── RETURNS ─────────────────────────────': '',
    'Total Return':           f'{df["Cum_Return"].iloc[-1]*100:.1f}%',
    'Annual Return (CAGR)':   f'{annual_return*100:.1f}%',
    '── RISK ────────────────────────────────': '',
    'Annual Volatility':      f'{annual_vol*100:.1f}%',
    'Sharpe Ratio':           f'{sharpe:.2f}',
    'Max Drawdown':           f'{max_dd*100:.1f}%',
    'VaR 95% (daily)':        f'{var_95*100:.2f}%',
    '── MONTE CARLO (1-year) ─────────────────': '',
    'Bear Case (5th pct)':    f'${p5:,.2f}',
    'Base Case (Median)':     f'${p50:,.2f}',
    'Bull Case (95th pct)':   f'${p95:,.2f}',
}

for k, v in summary.items():
    if v == '':
        print(f'\n  {k}')
    else:
        print(f'  {k:<35} {v}')

print('\n' + '═'*50)
print('  ✅ Analysis Complete')
print('═'*50)

---
> ⚠️ **Disclaimer:** This notebook is for **educational purposes only** and does not constitute financial advice. Past performance is not indicative of future results. Always conduct your own research before making any investment decisions.